# Вероятность опоздания более чем на 2 минуты
Модель выдаёт число для оператора: `P(факт прибытия > план + 120 секунд)`. Никакого порога для вывода класса здесь нет. Для оценки достоверности процента используются Brier score, log loss и таблица калибровки.

Обучение и калибровка разделены по семействам ТС: синтетические копии всегда остаются вместе с исходным ТС. Отложенная проверка проводится только на реальных ТС.


In [ ]:
from pathlib import Path
from mos_trans.modeling.calibrated_probability import run

output = Path('artifacts/calibrated_probability')
report = run('data/processed', 'data/dataset.zip', output)


In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(report['scores']).T)
print('Метод для оператора:', report['selected_method'])
display(pd.read_csv(output / f"reliability_{report['selected_method']}.csv"))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], '--', color='gray', label='идеальная калибровка')
for method in ('raw', 'sigmoid', 'isotonic'):
    table = pd.read_csv(output / f'reliability_{method}.csv')
    table = table[table['n'] > 0]
    ax.plot(table['mean_shown'], table['actual_rate'], 'o-', label=method)
ax.set(xlabel='Средняя показанная вероятность', ylabel='Реальная доля опозданий', xlim=(0, 1), ylim=(0, 1))
ax.legend(); ax.grid(alpha=.3)
plt.show()


In [ ]:
import joblib
from mos_trans.modeling.calibrated_probability import predict

bundle = joblib.load(output / 'model.joblib')
new_points = pd.read_parquet('data/processed/validate_features.parquet')
probabilities = predict(bundle, new_points)
display(pd.DataFrame({'sample_id': new_points.sample_id, 'probability_delay_over_120s': probabilities}).head())


## Как читать результат
Строка с вероятностью 0,8 означает оценку 80% для опоздания **более чем на две минуты** на конкретной целевой остановке. Смотрите число наблюдений `n` в соответствующей группе таблицы калибровки: при малом `n` фактическую долю нельзя оценить точно. Данные содержат один день и 13 реальных семейств ТС, поэтому перед применением на новых днях нужна отдельная проверка.
